In [ ]:
%pip install -q opencv-python-headless pillow numpy pandas ipywidgets
%pip install -q gradio
%pip install -q deepface
%pip install -q tf-keras

#NEW
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
%pip install --no-cache-dir opencv-python-headless

In [ ]:
from pathlib import Path

# Locate the shared project folder
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "Enrollment":
    ENROLLMENT_DIR = PROJECT_DIR
else:
    ENROLLMENT_DIR = PROJECT_DIR / "Enrollment"

DATABASE = ENROLLMENT_DIR / "students.db"
face_folder = ENROLLMENT_DIR / "registered_faces"

face_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
print("Database:", DATABASE)
print("Database exists:", DATABASE.is_file())
print("Registered faces folder:", face_folder)

In [ ]:
import cv2
print(cv2.__version__)
print(hasattr(cv2, "CascadeClassifier"))

In [ ]:
import cv2
import sqlite3
import numpy as np
import pandas as pd
import ipywidgets as widgets

from PIL import Image, ImageOps
from io import BytesIO
from pathlib import Path
from IPython.display import display

# Folder for registered student faces
face_folder = ENROLLMENT_DIR / "registered_faces"
face_folder.mkdir(exist_ok=True)

# Student database
conn = sqlite3.connect(DATABASE)

conn.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id TEXT PRIMARY KEY,
    student_name TEXT,
    face_image TEXT
)
""")

conn.commit()
conn.close()

# Pretrained face detector
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [ ]:
from deepface import DeepFace
from PIL import Image
from io import BytesIO
import numpy as np

def crop_face(image_bytes):
    image = Image.open(BytesIO(image_bytes)).convert("RGB")
    image_array = np.array(image)

    faces = DeepFace.extract_faces(
        img_path=image_array,
        detector_backend="opencv",
        enforce_detection=True,
        align=True
    )

    if len(faces) != 1:
        raise ValueError("Please upload an image with one clear face.")

    face = faces[0]["facial_area"]
    x, y, w, h = face["x"], face["y"], face["w"], face["h"]

    return image.crop((x, y, x + w, y + h))

In [ ]:
from uuid import uuid4

def register_student(student_id, student_name, image_bytes):
    student_id = student_id.strip()
    student_name = student_name.strip()

    if not student_id or not student_name:
        raise ValueError("Please enter student ID and name.")

    if not image_bytes:
        raise ValueError("Please upload a face image.")

    conn = sqlite3.connect(DATABASE)

    try:
        existing = conn.execute(
            "SELECT student_id FROM students WHERE student_id = ?",
            (student_id,)
        ).fetchone()

        if existing:
            raise ValueError("This student is already registered.")

        face = crop_face(image_bytes)

        image_path = face_folder / f"{uuid4().hex}.jpg"
        face.save(image_path)

        try:
            conn.execute(
                "INSERT INTO students VALUES (?, ?, ?)",
                (student_id, student_name, str(image_path))
            )
            conn.commit()
        except Exception:
            image_path.unlink(missing_ok=True)
            raise

        return face

    finally:
        conn.close()

In [ ]:
import subprocess
import gradio as gr
from io import BytesIO

def enroll_student(student_id, student_name, photo):
    try:
        if photo is None:
            return "Please upload or take a face photo."

        image_bytes = BytesIO()
        photo.save(image_bytes, format="JPEG")

        register_student(
            student_id,
            student_name,
            image_bytes.getvalue()
        )

        recognition_dir = ENROLLMENT_DIR.resolve().parent / "FaceRecognition"

        try:
            subprocess.run(
                [
                    str(recognition_dir / ".venv" / "bin" / "python"),
                    str(recognition_dir / "create_enrollment_gallery.py"),
                ],
                cwd=recognition_dir,
                check=True,
            )
        except (OSError, subprocess.CalledProcessError):
            return "Student registered, but the gallery refresh failed. Check the notebook output."

        return "Student registered and gallery rebuilt."

    except Exception as error:
        return str(error)


with gr.Blocks(title="Student Enrollment") as website:
    gr.Markdown("# Class Attendance System")
    gr.Markdown("### Student Enrollment")

    student_id = gr.Textbox(label="Student ID")
    student_name = gr.Textbox(label="Full Name")

    face_image = gr.Image(
        label="Face Photo",
        sources=["webcam", "upload"],
        type="pil"
    )

    register_button = gr.Button("Register Student", variant="primary")
    result = gr.Textbox(label="Registration Status", interactive=False)

    register_button.click(
        fn=enroll_student,
        inputs=[student_id, student_name, face_image],
        outputs=result
    )

website.launch(share=True)

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

def get_registered_students():
    conn = sqlite3.connect(DATABASE)

    students = pd.read_sql_query(
        "SELECT student_id, student_name, face_image FROM students",
        conn
    )

    conn.close()
    return students

registered_students = get_registered_students()

display(registered_students)